# nn-module-subclass — ex3: Linear layer from scratch

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `nn-module-subclass`. Running the final beacon cell reports progress against the `PyTorch: nn.Module subclassing` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: nn.Module subclassing` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`nn-module-subclass`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "nn-module-subclass"
DD_SUBTOPIC = "PyTorch: nn.Module subclassing"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `nn.Module` subclassing — quick refresher

Every learnable building block in PyTorch is an `nn.Module` subclass. The minimal pattern is:

```python
class MyLayer(nn.Module):
    def __init__(self, ...):
        super().__init__()      # MUST be first — wires up _parameters / _modules dicts
        self.weight = nn.Parameter(...)
    def forward(self, x):
        return ...              # never call .forward() directly — use module(x)
```

**Two non-obvious rules.**
1. If `__init__` does anything (assigns Parameters, sub-Modules, or buffers), it MUST call `super().__init__()` first. Forgetting this raises `AttributeError: cannot assign parameter before Module.__init__() call`.
2. Modules with no state can omit `__init__` entirely and just define `forward` (e.g. ARENA's `ReLU`). The base `nn.Module.__init__` runs implicitly.

**Call convention.** Use `module(x)`, never `module.forward(x)` — the `__call__` wrapper runs hooks (pre/post forward, gradient hooks) that you lose by calling forward directly.

### Exercise 3 — Linear layer from scratch

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Create
> LO: Build an end-to-end nn.Module with __init__ + super + forward that implements a Linear layer, returning the correct shape.
> Keywords: Linear, forward, matmul, bias
> ```

**KCs targeted:** `module-forward-signature`, `module-init-super-call`

Implement `MyLinear` — a from-scratch `nn.Linear` clone (the weights are passed in so we focus on the Module mechanics, NOT initialization):

1. `class MyLinear(t.nn.Module)` with `__init__(self, weight, bias)`:
   - Call `super().__init__()` first.
   - Wrap and store the inputs as Parameters: `self.weight = nn.Parameter(weight)`, `self.bias = nn.Parameter(bias)`.
2. `forward(self, x: Tensor) -> Tensor` computes `x @ self.weight.T + self.bias`.
   - `x` has shape `(*batch, in_features)`.
   - `self.weight` has shape `(out_features, in_features)` (matching `nn.Linear`'s convention).
   - `self.bias` has shape `(out_features,)`.
   - Output has shape `(*batch, out_features)`.

The function `ex3_build_linear(weight, bias)` should return an instance of `MyLinear` built with the supplied tensors.

**Don't reinvent initialization** — the test passes specific weight + bias tensors so we can check the forward output by hand.

In [ ]:
def ex3_build_linear(weight: Tensor, bias: Tensor):
    class MyLinear(t.nn.Module):
        def __init__(self, weight, bias):
            super().__init__()
            self.weight = t.nn.Parameter(weight)
            self.bias = t.nn.Parameter(bias)
        def forward(self, x: Tensor) -> Tensor:
            return x @ self.weight.T + self.bias
    return MyLinear(weight, bias)


<details><summary>Solution</summary>

```python
def ex3_build_linear(weight: Tensor, bias: Tensor):
    class MyLinear(t.nn.Module):
        def __init__(self, weight, bias):
            super().__init__()
            self.weight = t.nn.Parameter(weight)
            self.bias = t.nn.Parameter(bias)
        def forward(self, x: Tensor) -> Tensor:
            return x @ self.weight.T + self.bias
    return MyLinear(weight, bias)
```

**Why `weight.T` not `weight`.** PyTorch's `nn.Linear` stores weight as `(out_features, in_features)` — the transpose of what you'd write in a math textbook for `y = Wx` — so that `weight @ x` lines up dimensionally for a single sample but for batched input we want `x @ weight.T` (shape `(B, in) @ (in, out) = (B, out)`). Matching this convention means your `MyLinear` can swap in for `nn.Linear` (e.g. load its `state_dict`).

**Why bias broadcasts.** `self.bias` is `(out_features,)`. Adding it to a `(*batch, out_features)` tensor broadcasts over all leading batch dims for free — no `unsqueeze` needed.

**Leading-dim flexibility comes from `@`.** The `@` operator treats everything but the last two axes as batch dims, so `(B, S, in) @ (in, out) → (B, S, out)` works without any reshape. This is one reason real PyTorch code prefers `@` over `torch.matmul` calls + explicit reshaping.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()